# EdgeBPECoarsener example

This notebook learns attachment-independent label-pair rules, inspects the
schema-1 rule vocabulary, and checks an exact transform/decode round trip.


In [ ]:
from collections import Counter

from tree_coarsening import EdgeBPECoarsener
from tree_coarsening.utils import make_edge_bpe_dataset

training = make_edge_bpe_dataset(
    n_graphs=3,
    n_repeats=6,
    motif_labels=("A", "B", "C", "D"),
    seed=0,
)
raw = training[0]
print(f"training graphs: {len(training)}")
print(f"first raw tree: {raw.number_of_nodes()} nodes, {raw.number_of_edges()} edges")

In [ ]:
raw_pair_counts = Counter(
    (raw.nodes[parent]["label"], raw.nodes[child]["label"]) for parent, child in raw.edges
)
print(raw_pair_counts.most_common(8))

In [ ]:
model = EdgeBPECoarsener(
    num_merges=3,
    min_pair_count=4,
    pair_score="count",
    model_id="bpe-example",
).fit(training)

for step in model.history_:
    print(
        f"rank={step['rank']} | {step['parent_label']!r} -> {step['child_label']!r} | "
        f"raw count={step['count']} | contracted={step['actual_events']} | "
        f"new label={step['token']!r}"
    )

In [ ]:
print("Learned matching labels and fitting sizes:")
for label in model.encoder_.vocab.labels:
    print(label, model.encoder_.vocab.fitting_size(label))

In [ ]:
encoded = model.transform(raw)
print(f"raw nodes: {raw.number_of_nodes()}")
print(f"encoded nodes: {encoded.number_of_nodes()}")

for node, data in encoded.nodes(data=True):
    print(f"Node {node}: {data}")

In [ ]:
def raw_signature(graph):
    nodes = sorted(
        (data["uid"], tuple(sorted(data.items(), key=lambda item: repr(item[0]))))
        for _, data in graph.nodes(data=True)
    )
    edges = sorted(
        (graph.nodes[parent]["uid"], graph.nodes[child]["uid"]) for parent, child in graph.edges
    )
    return nodes, edges


roundtrip = model.decode(encoded)
assert raw_signature(roundtrip) == raw_signature(raw)
print("roundtrip ok")